In [1]:
import pandas as pd
import numpy as np
import rdkit
from rdkit import Chem

In [15]:
df = pd.read_csv('core_smarts.csv')
df.rename(columns={'First(First(Core))': 'Smarts'}, inplace=True)
# smarts to mol and set core index property from dataframe column
df.sort_values('Core index', inplace=True)
mol_list = []
for idx, row in df.iterrows():
    mol = Chem.MolFromSmarts(row['Smarts'])
    mol.SetProp('Core index', str(row['Core index']))
    mol_list.append(mol)

In [27]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import re

# visualize molecules in a grid with legends
# also set color for each legend based on core index

# Custom colors based on the user's palette (Set3-like colors)
custom_colors = [
    '#FB8072', # core_1: Red
    '#B3DE69', # core_2: Green
    '#80B1D3', # core_3: Blue
    '#FDB462', # core_4: Orange
    '#8DD3C7', # core_5: Teal
    '#BEBADA', # core_6: Lavender
    '#FFFFB3', # core_7: Yellow
    '#CCEBC5', # core_8: Pale Green
    '#D9D9D9', # core_9: Grey
    '#BC80BD', # core_10: Plum
]

# Generate colors for each unique core index
# Sort cores numerically to match colors to core_1, core_2, etc.
unique_cores = sorted(df['Core index'].unique(), key=lambda x: int(re.search(r'\d+', str(x)).group()))

# Map cores to colors
core_colors = {}
for i, core in enumerate(unique_cores):
    if i < len(custom_colors):
        core_colors[str(core)] = custom_colors[i]
    else:
        # Fallback if more cores than colors
        core_colors[str(core)] = '#000000'

# Function to draw SVG
def mol_to_svg(mol, width=150, height=100):
    d = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = d.drawOptions()
    opts.padding = 0.0 # Remove padding so molecule fills the canvas
    d.DrawMolecule(mol)
    d.FinishDrawing()
    return d.GetDrawingText()

html_content = '<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">'
for mol in mol_list:
    core_idx = mol.GetProp('Core index')
    svg = mol_to_svg(mol)
    color = core_colors.get(core_idx, 'black')
    
    html_content += f'''
    <div style="text-align: center; border: 1px solid #ddd; padding: 10px;">
        {svg}
        <div style="color: {color}; font-weight: bold; font-family: sans-serif; margin-top: 5px;">{core_idx}</div>
    </div>
    '''
html_content += '</div>'

display(HTML(html_content))